# Lab 2: Iterative Optimization & Gradient Descent

In Machine Learning, we often cannot solve for the optimal parameters analytically. Instead, we use numerical search techniques to find the minimum of our loss function. 

Before we jump into the math of Neural Networks, we are going to look at the geometry of how three famous algorithms work:
1. **Newton-Raphson:** Finding where a line crosses zero.
2. **Newton's Method:** Finding the absolute bottom of a valley.
3. **Gradient Descent:** The algorithm that powers modern AI.

In [29]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# 1. Define our landscape (Objective Function)
def f(x):
    return x**4 - 4 * x**2 + x + 3


# 2. Define the 1st derivative (The slope)
def df(x):
    return 4 * x**3 - 8 * x + 1


# 3. Define the 2nd derivative (The curvature)
def ddf(x):
    return 12 * x**2 - 8


# 4. Create standard x values for drawing the background curves
x_vals = np.linspace(-3, 3, 200)

print("Setup complete! Functions f(x), df(x), and ddf(x) are ready.")

Setup complete! Functions f(x), df(x), and ddf(x) are ready.


## Part 1: Root Finding (Newton-Raphson)

Before we optimize anything, we need to solve a simpler problem: How does a computer find where a function crosses zero? 

We use a technique called the **Newton-Raphson method**. If we want to find where our objective function $f(x) = 0$, we start at a random guess, calculate the slope $f'(x)$ at that point, and draw a tangent line down to the x-axis. That intercept becomes our new guess. 

Mathematically, the update rule is: 
$$x^{k+1} = x^k - \frac{f(x^k)}{f'(x^k)}$$

*Drag the slider to pick a starting point, let go, and press Play to watch it find the root!*

In [30]:
import ipywidgets as widgets
from IPython.display import display


def interactive_root_finding(x_start):
    iters = 6
    history_x = [x_start]

    # 1. Calculate the Newton-Raphson path for f(x) = 0
    for _ in range(iters):
        history_x.append(history_x[-1] - (f(history_x[-1]) / df(history_x[-1])))

    # 2. Build the static background
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=x_vals,
            y=f(x_vals),
            name="Objective f(x)",
            line=dict(color="lightgray", width=3),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[-3, 3],
            y=[0, 0],
            mode="lines",
            name="Zero Line",
            line=dict(color="black", dash="dash"),
        )
    )

    # 3. Add the initial starting point
    fig.add_trace(
        go.Scatter(
            x=[history_x[0]],
            y=[f(history_x[0])],
            mode="markers",
            name="Newton Steps",
            marker=dict(color="red", size=12),
        )
    )

    # 4. Create the Animation Frames
    frames = []
    for k in range(iters + 1):
        frames.append(
            go.Frame(
                data=[
                    go.Scatter(
                        x=history_x[: k + 1],
                        y=[f(x) for x in history_x[: k + 1]],
                        mode="markers+lines",
                        line=dict(color="red", width=2),
                    )
                ],
                traces=[2],
                name=f"{k}",
            )
        )
    fig.frames = frames

    # 5. Play Controls & Slider UI (Fixed Layout)
    slider_steps = [
        dict(
            method="animate",
            args=[
                [f"{k}"],
                dict(
                    mode="immediate",
                    frame=dict(duration=400, redraw=False),
                    transition=dict(duration=0),
                ),
            ],
            label=f"{k}",
        )
        for k in range(iters + 1)
    ]

    fig.update_layout(
        height=600,
        margin=dict(t=50, b=120),  # Extra bottom margin for controls
        updatemenus=[
            dict(
                type="buttons",
                direction="left",
                showactive=False,
                x=0.0,
                y=-0.15,
                xanchor="left",
                yanchor="top",
                pad=dict(t=0, r=10),
                buttons=[
                    dict(
                        label="▶ Play",
                        method="animate",
                        args=[
                            None,
                            dict(
                                frame=dict(duration=600, redraw=False),
                                fromcurrent=True,
                                transition=dict(duration=300),
                            ),
                        ],
                    ),
                    dict(
                        label="⏸ Pause",
                        method="animate",
                        args=[
                            [None],
                            dict(
                                frame=dict(duration=0, redraw=False),
                                mode="immediate",
                                transition=dict(duration=0),
                            ),
                        ],
                    ),
                ],
            )
        ],
        sliders=[
            dict(
                active=0,
                x=0.15,
                y=-0.15,
                xanchor="left",
                yanchor="top",
                pad=dict(t=0),
                currentvalue=dict(prefix="Iteration: ", font=dict(size=12)),
                steps=slider_steps,
            )
        ],
        template="plotly_white",
        title="Part 1: Finding the Root where f(x) = 0",
    )
    fig.show()


# Create the ipywidget slider
slider_root = widgets.FloatSlider(
    value=-2.5,
    min=-3.0,
    max=3.0,
    step=0.1,
    description="Start (x0):",
    continuous_update=False,
    layout=widgets.Layout(width="600px"),
)
out_root = widgets.interactive_output(
    interactive_root_finding, {"x_start": slider_root}
)
display(slider_root, out_root)

FloatSlider(value=-2.5, continuous_update=False, description='Start (x0):', layout=Layout(width='600px'), max=…

Output()

## Part 2: Optimization (Newton's Method for Minima)

Now, let's switch to optimization! We don't want to find where $f(x) = 0$ anymore; we want to find the bottom of the valley (the minimum). From calculus, we know the minimum occurs where the slope is exactly zero ($f'(x) = 0$).

Therefore, finding the minimum of $f(x)$ is exactly the same problem as finding the root of $f'(x)$! If we substitute $f'(x)$ into our root-finding formula from Part 1, we get **Newton's Method for Optimization**:
$$x^{k+1} = x^k - \frac{f'(x^k)}{f''(x^k)}$$

This method uses the second derivative $f''(x)$ (the curvature) to calculate the exact distance to leap to the bottom of the valley.

In [31]:
def interactive_newton_opt(x_start):
    iters = 6
    history_x = [x_start]

    # 1. Calculate the Newton Optimization path
    for _ in range(iters):
        history_x.append(history_x[-1] - (df(history_x[-1]) / ddf(history_x[-1])))

    # 2. Build the static background
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=x_vals,
            y=f(x_vals),
            name="Objective f(x)",
            line=dict(color="lightgray", width=3),
        )
    )

    # 3. Add the initial starting point
    fig.add_trace(
        go.Scatter(
            x=[history_x[0]],
            y=[f(history_x[0])],
            mode="markers",
            name="Newton Jumps",
            marker=dict(color="red", symbol="diamond", size=12),
        )
    )

    # 4. Create the Animation Frames
    frames = []
    for k in range(iters + 1):
        frames.append(
            go.Frame(
                data=[
                    go.Scatter(
                        x=history_x[: k + 1],
                        y=[f(x) for x in history_x[: k + 1]],
                        mode="markers+lines",
                        line=dict(color="red", width=2, dash="dot"),
                    )
                ],
                traces=[1],
                name=f"{k}",
            )
        )
    fig.frames = frames

    # 5. Play Controls & Slider UI (Fixed Layout)
    slider_steps = [
        dict(
            method="animate",
            args=[
                [f"{k}"],
                dict(
                    mode="immediate",
                    frame=dict(duration=400, redraw=False),
                    transition=dict(duration=0),
                ),
            ],
            label=f"{k}",
        )
        for k in range(iters + 1)
    ]

    fig.update_layout(
        height=600,
        margin=dict(t=50, b=120),
        updatemenus=[
            dict(
                type="buttons",
                direction="left",
                showactive=False,
                x=0.0,
                y=-0.15,
                xanchor="left",
                yanchor="top",
                pad=dict(t=0, r=10),
                buttons=[
                    dict(
                        label="▶ Play",
                        method="animate",
                        args=[
                            None,
                            dict(
                                frame=dict(duration=600, redraw=False),
                                fromcurrent=True,
                                transition=dict(duration=300),
                            ),
                        ],
                    ),
                    dict(
                        label="⏸ Pause",
                        method="animate",
                        args=[
                            [None],
                            dict(
                                frame=dict(duration=0, redraw=False),
                                mode="immediate",
                                transition=dict(duration=0),
                            ),
                        ],
                    ),
                ],
            )
        ],
        sliders=[
            dict(
                active=0,
                x=0.15,
                y=-0.15,
                xanchor="left",
                yanchor="top",
                pad=dict(t=0),
                currentvalue=dict(prefix="Iteration: ", font=dict(size=12)),
                steps=slider_steps,
            )
        ],
        template="plotly_white",
        title="Part 2: Newton's Method (Using Curvature to Leap)",
    )
    fig.show()


slider_opt = widgets.FloatSlider(
    value=2.5,
    min=-3.0,
    max=3.0,
    step=0.1,
    description="Start (x0):",
    continuous_update=False,
    layout=widgets.Layout(width="600px"),
)
out_opt = widgets.interactive_output(interactive_newton_opt, {"x_start": slider_opt})
display(slider_opt, out_opt)

FloatSlider(value=2.5, continuous_update=False, description='Start (x0):', layout=Layout(width='600px'), max=3…

Output()

## Part 3: Gradient Descent (Steepest Descent)

Newton's method is incredibly fast, but computing the second derivative (the Hessian matrix) for millions of machine learning parameters requires massive computational power. 

Instead, modern AI uses **Gradient Descent**. We throw away the expensive $\frac{1}{f''(x^k)}$ curvature term and replace it with a small, cheap, fixed constant called the **learning rate**, $\alpha$. 
$$x^{k+1} = x^k - \alpha f'(x^k)$$

Because the algorithm no longer knows the exact curvature, it has to take smaller, greedy "steepest descent" steps down the slope. 

*Try starting at $x_0 = -0.5$ versus $x_0 = 0.5$ to see how greedy algorithms get trapped in local minima!*

In [33]:
def interactive_gradient_descent(x_start):
    iters = 15
    alpha = 0.05  # Fixed learning rate
    history_x = [x_start]

    # 1. Calculate the Gradient Descent path
    for _ in range(iters):
        history_x.append(history_x[-1] - alpha * df(history_x[-1]))

    # 2. Build the static background
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=x_vals,
            y=f(x_vals),
            name="Objective f(x)",
            line=dict(color="lightgray", width=3),
        )
    )

    # 3. Add the initial starting point
    fig.add_trace(
        go.Scatter(
            x=[history_x[0]],
            y=[f(history_x[0])],
            mode="markers",
            name="Gradient Descent",
            marker=dict(color="blue", size=12),
        )
    )

    # 4. Create the Animation Frames
    frames = []
    for k in range(iters + 1):
        frames.append(
            go.Frame(
                data=[
                    go.Scatter(
                        x=history_x[: k + 1],
                        y=[f(x) for x in history_x[: k + 1]],
                        mode="markers+lines",
                        line=dict(color="blue", width=2),
                    )
                ],
                traces=[1],
                name=f"{k}",
            )
        )
    fig.frames = frames

    # 5. Play Controls & Slider UI (Fixed Layout)
    slider_steps = [
        dict(
            method="animate",
            args=[
                [f"{k}"],
                dict(
                    mode="immediate",
                    frame=dict(duration=200, redraw=False),
                    transition=dict(duration=0),
                ),
            ],
            label=f"{k}",
        )
        for k in range(iters + 1)
    ]

    fig.update_layout(
        height=600,
        margin=dict(t=50, b=120),
        updatemenus=[
            dict(
                type="buttons",
                direction="left",
                showactive=False,
                x=0.0,
                y=-0.15,
                xanchor="left",
                yanchor="top",
                pad=dict(t=0, r=10),
                buttons=[
                    dict(
                        label="▶ Play",
                        method="animate",
                        args=[
                            None,
                            dict(
                                frame=dict(duration=300, redraw=False),
                                fromcurrent=True,
                                transition=dict(duration=150),
                            ),
                        ],
                    ),
                    dict(
                        label="⏸ Pause",
                        method="animate",
                        args=[
                            [None],
                            dict(
                                frame=dict(duration=0, redraw=False),
                                mode="immediate",
                                transition=dict(duration=0),
                            ),
                        ],
                    ),
                ],
            )
        ],
        sliders=[
            dict(
                active=0,
                x=0.15,
                y=-0.15,
                xanchor="left",
                yanchor="top",
                pad=dict(t=0),
                currentvalue=dict(prefix="Iteration: ", font=dict(size=12)),
                steps=slider_steps,
            )
        ],
        template="plotly_white",
        title=f"Part 3: Gradient Descent (alpha = {alpha})",
    )
    fig.show()


slider_gd = widgets.FloatSlider(
    value=2.5,
    min=-3.0,
    max=3.0,
    step=0.1,
    description="Start (x0):",
    continuous_update=False,
    layout=widgets.Layout(width="600px"),
)
out_gd = widgets.interactive_output(
    interactive_gradient_descent, {"x_start": slider_gd}
)
display(slider_gd, out_gd)

FloatSlider(value=2.5, continuous_update=False, description='Start (x0):', layout=Layout(width='600px'), max=3…

Output()

## The Grand Finale: The Algorithm Race

Now that we understand all three concepts, let's put them all on the **exact same plot** to watch them race! 

1. **Newton's Root Finder (Green):** Is trying to find where the curve crosses the dashed zero-line ($f(x) = 0$).
2. **Newton's Optimizer (Red):** Is using curvature to leap to the exact bottom of the valley (min $f(x)$).
3. **Gradient Descent (Blue):** Is taking timid steps down the slope to find the bottom (min $f(x)$).

**The Sandbox:**
Use the sliders below to change your starting position ($x_0$) and the Gradient Descent Learning Rate ($\alpha$). 
*Tip: Try turning the learning rate up to 0.12, hit play, and watch Gradient Descent violently bounce out of the valley!*

In [34]:
import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
from IPython.display import display


def interactive_grand_finale(x_start, alpha):
    iters = 15

    # 1. Recalculate paths based on the NEW x_start and alpha
    x_nt_root = [x_start]  # Newton Root Finder
    x_nt_opt = [x_start]  # Newton Optimizer
    x_gd = [x_start]  # Gradient Descent

    for _ in range(iters):
        # Root Finder: x - f(x)/f'(x)
        d_val = df(x_nt_root[-1])
        if abs(d_val) < 1e-5:
            d_val = 1e-5 if d_val >= 0 else -1e-5
        x_nt_root.append(x_nt_root[-1] - (f(x_nt_root[-1]) / d_val))

        # Optimizer: x - f'(x)/f''(x)
        dd_val = ddf(x_nt_opt[-1])
        if abs(dd_val) < 1e-5:
            dd_val = 1e-5 if dd_val >= 0 else -1e-5
        x_nt_opt.append(x_nt_opt[-1] - (df(x_nt_opt[-1]) / dd_val))

        # Gradient Descent: x - alpha * f'(x)
        x_gd.append(x_gd[-1] - alpha * df(x_gd[-1]))

    # 2. Build the Static Background
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=x_vals,
            y=f(x_vals),
            name="Objective f(x)",
            line=dict(color="lightgray", width=3),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[-3.5, 3.5],
            y=[0, 0],
            mode="lines",
            name="Zero Line",
            line=dict(color="black", dash="dash"),
        )
    )

    # 3. Initial Traces (Iteration 0)
    fig.add_trace(
        go.Scatter(
            x=[x_nt_root[0]],
            y=[f(x_nt_root[0])],
            mode="markers",
            name="Newton Root Finder",
            marker=dict(color="green", symbol="square", size=10),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[x_nt_opt[0]],
            y=[f(x_nt_opt[0])],
            mode="markers",
            name="Newton Optimizer",
            marker=dict(color="red", symbol="diamond", size=10),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[x_gd[0]],
            y=[f(x_gd[0])],
            mode="markers",
            name="Gradient Descent",
            marker=dict(color="blue", size=10),
        )
    )

    # 4. Create Animation Frames
    frames = []
    for k in range(iters + 1):
        frames.append(
            go.Frame(
                data=[
                    go.Scatter(
                        x=x_nt_root[: k + 1],
                        y=[f(x) for x in x_nt_root[: k + 1]],
                        mode="markers+lines",
                        line=dict(color="green", width=2, dash="dot"),
                    ),
                    go.Scatter(
                        x=x_nt_opt[: k + 1],
                        y=[f(x) for x in x_nt_opt[: k + 1]],
                        mode="markers+lines",
                        line=dict(color="red", width=2),
                    ),
                    go.Scatter(
                        x=x_gd[: k + 1],
                        y=[f(x) for x in x_gd[: k + 1]],
                        mode="markers+lines",
                        line=dict(color="blue", width=2),
                    ),
                ],
                traces=[2, 3, 4],
                name=f"{k}",  # Updating the 3 animated traces
            )
        )
    fig.frames = frames

    # 5. Play Controls & Slider UI
    slider_steps = [
        dict(
            method="animate",
            args=[
                [f"{k}"],
                dict(
                    mode="immediate",
                    frame=dict(duration=300, redraw=False),
                    transition=dict(duration=0),
                ),
            ],
            label=f"{k}",
        )
        for k in range(iters + 1)
    ]

    fig.update_layout(
        height=600,
        margin=dict(t=50, b=120),  # Gives space for the controls
        xaxis=dict(range=[-3.2, 3.2]),  # Lock X-axis so it doesn't shake
        yaxis=dict(range=[-10, 50]),  # Lock Y-axis to handle GD divergence gracefully
        updatemenus=[
            dict(
                type="buttons",
                direction="left",
                showactive=False,
                x=0.0,
                y=-0.15,
                xanchor="left",
                yanchor="top",
                pad=dict(t=0, r=10),
                buttons=[
                    dict(
                        label="▶ Play",
                        method="animate",
                        args=[
                            None,
                            dict(
                                frame=dict(duration=600, redraw=False),
                                fromcurrent=True,
                                transition=dict(duration=300),
                            ),
                        ],
                    ),
                    dict(
                        label="⏸ Pause",
                        method="animate",
                        args=[
                            [None],
                            dict(
                                frame=dict(duration=0, redraw=False),
                                mode="immediate",
                                transition=dict(duration=0),
                            ),
                        ],
                    ),
                ],
            )
        ],
        sliders=[
            dict(
                active=0,
                x=0.15,
                y=-0.15,
                xanchor="left",
                yanchor="top",
                pad=dict(t=0),
                currentvalue=dict(prefix="Iteration: ", font=dict(size=12)),
                steps=slider_steps,
            )
        ],
        template="plotly_white",
        legend=dict(yanchor="top", y=0.99, xanchor="center", x=0.5, orientation="h"),
    )
    fig.show()


# 6. Create the two ipywidget sliders
slider_x0 = widgets.FloatSlider(
    value=2.5,
    min=-3.0,
    max=3.0,
    step=0.1,
    description="Start (x0):",
    continuous_update=False,
    layout=widgets.Layout(width="400px"),
)

slider_alpha = widgets.FloatSlider(
    value=0.03,
    min=0.005,
    max=0.15,
    step=0.005,
    description="Learning Rate (α):",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)

# Bind the sliders to the function and display them side-by-side
ui = widgets.HBox([slider_x0, slider_alpha])
out_finale = widgets.interactive_output(
    interactive_grand_finale, {"x_start": slider_x0, "alpha": slider_alpha}
)

display(ui, out_finale)

Output()